In [2]:
import collections
import re
from d2l import torch as d2l
import os

In [3]:
#@save
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                        '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine(): #@save
    # 将《时间机器》数据集加载到文本行的列表中
    #with open(d2l.download('time_machine'),'r') as f:
    
    # 直接用本地路径，不让 d2l 去下载
    data_dir = '../data'  # 根据你实际位置调整
    file_path = os.path.join(data_dir, 'timemachine.txt')
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines() # 一次性读取文件的所有行，返回一个字符串列表，每个元素是文件中的一行文本（包含换行符）
    return [re.sub('[^A-Za-z]+', ' ',line).strip().lower() for line in lines] # 对 lines 中的每一行逐个处理，处理完组成一个新列表返回，利用正则变换 将非字母转换为空格
                                                                              #首尾的空白字符（空格、换行等），避免首尾有多余空格，把所有大写字母转成小写，统一大小写
lines= read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])

# 文本总行数: 3224
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


In [8]:
# 词元化
"""tokenize函数将文本行列表（lines）作为输入，列表中的每个元素是一个文本序列（如一条文本行）。
每个文本序列又被拆分成一个词元列表，词元（token）是文本的基本单位。最后，返回一个由词元列表组成
的列表，其中的每个词元都是一个字符串（string）"""

'tokenize函数将文本行列表（lines）作为输入，列表中的每个元素是一个文本序列（如一条文本行）。\n每个文本序列又被拆分成一个词元列表，词元（token）是文本的基本单位。最后，返回一个由词元列表组成\n的列表，其中的每个词元都是一个字符串（string）'

In [28]:
def tokenize(lines,token='word'):
    #将文本拆分为单词或字符词元
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型: '+ token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


In [29]:
# 词表
"""建一个字典，通常也叫做词表（vocabulary），用来将字符串类型的词元映射到从0开始的数字索引中。我们先将训练
集中的所有文档合并在一起，对它们的唯一词元进行统计，得到的统计结果称之为语料（corpus）。然后根
据每个唯一词元的出现频率，为其分配一个数字索引。很少出现的词元通常被移除，这可以降低复杂性。另
外，语料库中不存在或已删除的任何词元都将映射到一个特定的未知词元“<unk>”。我们可以选择增加一个
列表，用于保存那些被保留的词元，例如：填充词元（“<pad>”）；序列开始词元（“<bos>”）；序列结束词元
（“<eos>”）"""

'建一个字典，通常也叫做词表（vocabulary），用来将字符串类型的词元映射到从0开始的数字索引中。我们先将训练\n集中的所有文档合并在一起，对它们的唯一词元进行统计，得到的统计结果称之为语料（corpus）。然后根\n据每个唯一词元的出现频率，为其分配一个数字索引。很少出现的词元通常被移除，这可以降低复杂性。另\n外，语料库中不存在或已删除的任何词元都将映射到一个特定的未知词元“<unk>”。我们可以选择增加一个\n列表，用于保存那些被保留的词元，例如：填充词元（“<pad>”）；序列开始词元（“<bos>”）；序列结束词元\n（“<eos>”）'

In [30]:
class Vocab:
    #文本词表
    def __init__(self,tokens=None,min_freq=0,reserved_tokens=None):
        if tokens is None: 
            tokens = []
        if reserved_tokens is None: #要保留的特殊词，最后Vocab 把两者合并成一套完整的词表，普通词按频率排，特殊词固定在开头
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], #自定义顺序排序，按出现频率
                                    reverse=True) #降序
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        
        self.token_to_idx = {token: idx
                            for idx, token in enumerate(self.idx_to_token)} #词元→索引的字典推导式。enumerate() 同时取出索引和词元
        
        for token, freq in self._token_freqs: #遍历之前按频率排好序的 (词元, 频率) 列表
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1
                
    def __len__(self):
        return len(self.idx_to_token)
        
    def __getitem__(self, tokens): #词元转索引 ，这个方法是 Python 的特殊方法，让你可以用 vocab[词] 这种方括号语法。
        if not isinstance(tokens, (list, tuple)): #判断一个对象是否属于某个类型，isinstance(对象, 类型)，tuple是不可变列表
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]
        
    def to_tokens(self, indices): #把索引转回对应的词元
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property # 把方法变成属性调用
    def unk(self): # 未知词元的索引为0
        return 0
            
    @property
    def token_freqs(self):
        return self._token_freqs
            
def count_corpus(tokens): #@save
    """统计词元的频率"""
    # 这里的tokens是1D列表或2D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将词元列表展平成一个列表
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)


In [31]:
vocab = Vocab(tokens)
print(list(vocab.token_to_idx.items())[:10])
print((vocab.idx_to_token)[:10])
print(vocab.token_freqs[:10])  # 查看前 10 个高频词及其频率

[('<unk>', 0), ('the', 1), ('i', 2), ('and', 3), ('of', 4), ('a', 5), ('to', 6), ('was', 7), ('in', 8), ('that', 9)]
['<unk>', 'the', 'i', 'and', 'of', 'a', 'to', 'was', 'in', 'that']
[('the', 2261), ('i', 1267), ('and', 1245), ('of', 1155), ('a', 816), ('to', 695), ('was', 552), ('in', 541), ('that', 443), ('my', 440)]


In [32]:
#将每一条文本行转换一个数字索引列表
for i in [0,10]: #这里只会取 0 和 10
    print('文本:', tokens[i]) #保存的 每一行词元的所有行，token[i] 表示每一行tokens
    print('索引:', vocab[tokens[i]])

文本: ['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
索引: [1, 19, 50, 40, 2183, 2184, 400]
文本: ['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']
索引: [2186, 3, 25, 1044, 362, 113, 7, 1421, 3, 1045, 1]


In [33]:
def load_corpus_time_machine(max_tokens=-1): #@save
    """返回时光机器数据集的词元索引列表和词表"""
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    # 因为时光机器数据集中的每个文本行不一定是一个句子或一个段落，
    # 所以将所有文本行展平到一个列表中
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab
    
corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)


(170580, 28)

In [ ]:
# 为了对文本进行预处理，我们通常将文本拆分为词元，构建词表将词元字符串映射为数字索引，并将文本数据转换为词元索引以供模型操作。